# Task 4 — CNN Classifier and Adversarial Images on MNIST

In this notebook, we:
1. Train a CNN classifier on MNIST
2. Create targeted adversarial images that force correctly classified
   digit 4s to be predicted as 9 (iterative FGSM)
3. Optimize a random noise image so the classifier predicts digit 9
4. Compare both results

## Adversarial attacks

A **targeted adversarial attack** adds a carefully crafted,
imperceptible perturbation to an input image to force a classifier
to output a chosen wrong class. We use an iterative signed-gradient
method (PGD-style): at each step, the gradient of the cross-entropy
loss is computed w.r.t. the *input pixels* (not the model weights),
and the image is shifted by a small step `alpha` in the direction
that minimises the target-class loss. The total perturbation is
clipped to an L∞ ball of radius `epsilon` around the original.

**Noise optimisation** applies the same idea to a random starting
image using Adam rather than signed gradient steps. Both approaches
are compared at the end.

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import wandb

import global_config as gc
from utils import device_check

In [ ]:
LOG_WANDB = True
SEED = 1

device = device_check()

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    PIN_MEMORY = True
    NUM_WORKERS = 8
    PERSISTENT_WORKERS = True
    PREFETCH_FACTOR = 4
else:
    PIN_MEMORY = False
    NUM_WORKERS = 0
    PERSISTENT_WORKERS = False
    PREFETCH_FACTOR = 2

## Load MNIST dataset

In [ ]:
BATCH_SIZE = 128

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(
    root=gc.DATA_DIR, train=True, transform=transform, download=True
)
test_dataset = datasets.MNIST(
    root=gc.DATA_DIR, train=False, transform=transform, download=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
)

print("Train samples:", len(train_dataset))
print("Test samples: ", len(test_dataset))

## Visualize a few real images

In [ ]:
images, labels = next(iter(test_loader))

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i in range(6):
    axes[i].imshow(images[i].squeeze(), cmap="gray")
    axes[i].set_title(f"Label: {labels[i].item()}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## Define the CNN classifier

In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

## Initialize model and optimizer

In [ ]:
LR = 1e-3
BETAS = (0.9, 0.999)

model = MNISTClassifier().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, betas=BETAS)

print(model)

## Train the classifier

In [ ]:
EPOCHS = 100
SOURCE_DIGIT = 4
TARGET_DIGIT = 9
EPSILON = 0.25
ATTACK_STEPS = 50
ATTACK_ALPHA = 0.01
NOISE_STEPS = 300
NOISE_LR = 0.05

config = {
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "betas": BETAS,
    "optimizer": "Adam",
    "dataset": "MNIST",
    "model": "CNN Classifier",
    "seed": SEED,
    "source_digit": SOURCE_DIGIT,
    "target_digit": TARGET_DIGIT,
    "epsilon": EPSILON,
    "attack_steps": ATTACK_STEPS,
    "attack_alpha": ATTACK_ALPHA,
    "noise_steps": NOISE_STEPS,
    "noise_lr": NOISE_LR,
}

wandb_kwargs = dict(
    entity=gc.WANDB_ENTITY,
    project=gc.WANDB_PROJECT,
    name="Task4 - CNN Adversarial MNIST",
    tags=["Task 4", "MNIST", "CNN", "Adversarial"],
    dir=str(gc.WANDB_DIR),
    config=config,
    mode="online" if LOG_WANDB else "disabled",
)

wandb.init(**wandb_kwargs)  # type: ignore[arg-type]

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda") if use_amp else None  # type: ignore[attr-defined]
criterion = nn.CrossEntropyLoss()

for epoch in tqdm(range(EPOCHS), desc="Training", unit="ep"):
    model.train()
    loss_sum = correct = total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        optimizer.zero_grad()
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        loss_sum += loss.item() * labels.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += labels.size(0)

    wandb.log(
        {"train_loss": loss_sum / total, "train_acc": correct / total},
        step=epoch + 1,
    )

## Evaluate on the test set

In [ ]:
model.eval()
correct = total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        preds = model(images).argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

test_acc = correct / total
print(f"Test accuracy: {test_acc:.4f}")
wandb.log({"test_acc": test_acc})

## Save trained classifier

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ckpt_name = (
    f"task4_classifier"
    f"_ep{EPOCHS}"
    f"_bs{BATCH_SIZE}"
    f"_lr{LR}"
    f"_seed{SEED}"
    f"_{timestamp}.pt"
)
ckpt_path = gc.MODELS_DIR / ckpt_name

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": config,
    },
    ckpt_path,
)
print(f"Saved: {ckpt_path}")

## Find correctly classified digit 4 images

In [ ]:
model.eval()
selected_images = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        preds  = model(images).argmax(1)
        mask   = (labels == SOURCE_DIGIT) & (preds == SOURCE_DIGIT)
        for img in images[mask]:
            selected_images.append(img.unsqueeze(0).clone())
        if len(selected_images) >= 10:
            break

selected_images = selected_images[:10]
print(
    f"Collected {len(selected_images)} correctly classified"
    f" digit-{SOURCE_DIGIT} images."
)

## Visualize selected digit 4 images

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, img in enumerate(selected_images):
    axes[i].imshow(img.squeeze().cpu().numpy(), cmap="gray")
    axes[i].set_title(str(SOURCE_DIGIT))
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## Targeted adversarial attack: digit 4 → digit 9

In [ ]:
def targeted_attack(
    model, image, target_label,
    epsilon=0.25, steps=50, alpha=0.01,
):
    """Iterative signed-gradient targeted attack (L∞ constraint)."""
    model.eval()
    original = image.detach().to(device)
    adv = original.clone().requires_grad_(True)
    target = torch.tensor([target_label], device=device)

    for _ in range(steps):
        loss = F.cross_entropy(model(adv), target)
        loss.backward()
        with torch.no_grad():
            adv_next = adv - alpha * adv.grad.sign()
            perturbation = torch.clamp(
                adv_next - original, -epsilon, epsilon
            )
            adv_next = torch.clamp(original + perturbation, 0.0, 1.0)
        adv = adv_next.requires_grad_(True)

    return adv.detach()

## Generate adversarial examples

In [ ]:
adv_images = []
adv_preds  = []

for img in selected_images:
    adv = targeted_attack(
        model, img, TARGET_DIGIT,
        epsilon=EPSILON, steps=ATTACK_STEPS, alpha=ATTACK_ALPHA,
    )
    with torch.no_grad():
        pred = model(adv).argmax(1).item()
    adv_images.append(adv)
    adv_preds.append(pred)

print("Predicted labels:", adv_preds)

## Show original vs adversarial images

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
for i in range(10):
    orig = selected_images[i].squeeze().cpu().numpy()
    adv  = adv_images[i].squeeze().cpu().numpy()
    delta = adv - orig
    # Normalize delta to [0, 1] for display
    delta_vis = (delta - delta.min()) / (delta.max() - delta.min() + 1e-8)

    axes[0, i].imshow(orig, cmap="gray")
    axes[0, i].set_title(f"Orig: {SOURCE_DIGIT}", fontsize=7)
    axes[0, i].axis("off")

    axes[1, i].imshow(delta_vis, cmap="gray")
    axes[1, i].set_title("Delta", fontsize=7)
    axes[1, i].axis("off")

    axes[2, i].imshow(adv, cmap="gray")
    axes[2, i].set_title(f"Pred: {adv_preds[i]}", fontsize=7)
    axes[2, i].axis("off")

plt.tight_layout()
plt.show()

## Optimize random noise to be classified as digit 9

In [ ]:
def optimize_noise(model, target_label, steps=300, lr=0.05):
    """Gradient descent on a random image to maximize classifier
    confidence for target_label."""
    model.eval()
    noise  = torch.rand(1, 1, 28, 28, device=device, requires_grad=True)
    opt    = optim.Adam([noise], lr=lr)
    target = torch.tensor([target_label], device=device)

    for _ in range(steps):
        loss = F.cross_entropy(model(noise), target)
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            noise.clamp_(0.0, 1.0)

    return noise.detach()


optimized_noise = optimize_noise(
    model, TARGET_DIGIT, steps=NOISE_STEPS, lr=NOISE_LR
)

with torch.no_grad():
    noise_logits = model(optimized_noise)
    noise_pred   = noise_logits.argmax(1).item()
    noise_conf   = torch.softmax(noise_logits, 1)[0, TARGET_DIGIT].item()

print(f"Predicted: {noise_pred}")
print(f"Confidence for digit {TARGET_DIGIT}: {noise_conf:.4f}")

## Visualize the optimized noise image

In [ ]:
plt.figure(figsize=(3, 3))
plt.imshow(optimized_noise.squeeze().cpu().numpy(), cmap="gray")
plt.title(f"Optimized noise \u2192 predicted: {noise_pred}")
plt.axis("off")
plt.show()

## Compare: original, adversarial, optimized noise

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))

axes[0].imshow(
    selected_images[0].squeeze().cpu().numpy(), cmap="gray"
)
axes[0].set_title(f"Original {SOURCE_DIGIT}")
axes[0].axis("off")

axes[1].imshow(adv_images[0].squeeze().cpu().numpy(), cmap="gray")
axes[1].set_title(f"Adversarial \u2192 {adv_preds[0]}")
axes[1].axis("off")

axes[2].imshow(optimized_noise.squeeze().cpu().numpy(), cmap="gray")
axes[2].set_title(f"Noise \u2192 {noise_pred}")
axes[2].axis("off")

plt.tight_layout()

adv_success = sum(p == TARGET_DIGIT for p in adv_preds) / len(adv_preds)
wandb.log({
    "comparison_figure": wandb.Image(
        fig, caption="Original / Adversarial / Noise"
    ),
    "adv_success_rate": adv_success,
    "noise_confidence": noise_conf,
})

plt.show()
wandb.finish()